In [31]:
import duckdb
import pandas as pd
import json
import numpy as np

# Connect to DuckDB
con = duckdb.connect(':memory:')

# First, let's see what columns we have in the parquet files
print("Schema of parquet files:")
schema = con.execute("DESCRIBE SELECT * FROM read_parquet('company_details/*.parquet', union_by_name=True)").df()
display(schema)

# Now let's examine the financials column structure
query = """
SELECT company_number, financials
FROM read_parquet('company_details/*.parquet', union_by_name=True)
WHERE financials IS NOT NULL
  AND length(financials) > 0
LIMIT 1;
"""

df = con.execute(query).df()

print("\nExamining financial data structure:")
if len(df) > 0:
    print(f"\nCompany number: {df['company_number'].iloc[0]}")
    fin_data = df['financials'].iloc[0]
    print(f"Type of financials: {type(fin_data)}")
    
    # Convert numpy types to Python native types for better display
    def convert_to_serializable(obj):
        if isinstance(obj, (np.int_, np.intc, np.intp, np.int8, np.int16, np.int32, np.int64, np.uint8, np.uint16, np.uint32, np.uint64)):
            return int(obj)
        elif isinstance(obj, (np.float_, np.float16, np.float32, np.float64)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, pd.NA):
            return None
        elif pd.isna(obj):
            return None
        return obj
    
    # Print raw structure
    print("\nRaw structure of first financial entry:")
    if isinstance(fin_data, (list, np.ndarray)):
        if len(fin_data) > 0:
            first_entry = convert_to_serializable(fin_data[0])
            print(json.dumps(first_entry, indent=2, default=str))
            
            # If there are facts, show the structure of the first fact
            if isinstance(first_entry, dict) and 'facts' in first_entry:
                print("\nStructure of first fact:")
                if len(first_entry['facts']) > 0:
                    print(json.dumps(first_entry['facts'][0], indent=2, default=str))
    else:
        print(f"Unexpected data type: {type(fin_data)}")

Sample financial data structure:


In [4]:
# Helper functions to process JSON columns
import numpy as np

def convert_numpy_to_python(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_numpy_to_python(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_to_python(item) for item in obj]
    return obj

def process_json_column(df, column_name):
    """Process a JSON column and return it as a pandas DataFrame with normalized data"""
    if column_name not in df.columns:
        print(f"Column {column_name} not found in the dataset")
        return None
    
    # Extract and normalize the JSON data
    processed_data = []
    
    for idx, row in df.iterrows():
        data = row[column_name]
        if isinstance(data, str):
            try:
                data = json.loads(data)
            except json.JSONDecodeError:
                continue
                
        # Convert NumPy types to Python native types
        data = convert_numpy_to_python(data)
        
        if isinstance(data, dict):
            processed_data.append(data)
        
    # Convert to DataFrame
    return pd.json_normalize(processed_data)

# Process the financials column as an example
if 'financials' in df.columns:
    financials_df = process_json_column(df, 'financials')
    if financials_df is not None:
        print("Financials Data Structure:")
        print("\nColumns in the financials data:")
        for col in financials_df.columns:
            print(f"- {col}")
        
        print("\nFinancials Data Preview:")
        display(financials_df.head())

[
  {
    "facts": [
      {
        "concept": "bus:UKCompaniesHouseRegisteredNumber",
        "context": "C",
        "period": "2023-04-01/2024-03-31",
        "unit": null,
        "value": 220603.0
      },
      {
        "concept": "core:PropertyPlantEquipment",
        "context": "B",
        "period": "2024-03-31",
        "unit": "GBP",
        "value": 2510.0
      },
      {
        "concept": "core:PropertyPlantEquipment",
        "context": "E",
        "period": "2023-03-31",
        "unit": "GBP",
        "value": 359.0
      },
      {
        "concept": "core:Debtors",
        "context": "B",
        "period": "2024-03-31",
        "unit": "GBP",
        "value": 4908.0
      },
      {
        "concept": "core:Debtors",
        "context": "E",
        "period": "2023-03-31",
        "unit": "GBP",
        "value": 1409.0
      },
      {
        "concept": "core:CashBankOnHand",
        "context": "B",
        "period": "2024-03-31",
        "unit": "GBP",
        "v

# Financial Facts Analysis

The financial facts in the data are structured as follows:
1. Each company can have multiple filings
2. Each filing has multiple financial facts
3. Each fact has:
   - concept: The type of financial measure
   - value: The numerical value
   - unit: The currency or unit of measurement
   - period: The time period it refers to
   - context: Additional context about the measurement

In [46]:
# First, let's get the financial facts from the parquet files
import plotly.express as px
import duckdb
import pandas as pd

# Connect to DuckDB in memory
con = duckdb.connect()

# First, let's examine the structure
inspect_query = """
WITH expanded AS (
    SELECT 
        company_number,
        UNNEST(financials) as filing
    FROM read_parquet('company_details/*.parquet', union_by_name=True)
    WHERE financials IS NOT NULL
    LIMIT 1
)
SELECT 
    company_number,
    filing.*,
    unnest.* 
FROM expanded,
UNNEST(filing.facts) AS unnest
LIMIT 1;
"""

# Execute inspection query to see the structure
print("Examining facts structure:")
inspection_result = con.execute(inspect_query).df()
print("\nAvailable columns in facts:")
display(inspection_result)

# Now let's modify our main query based on the actual structure
query = """
WITH financial_facts AS (
    SELECT 
        company_number,
        filing.filing_type,
        filing.filing_date,
        unnest.period as period,
        unnest.concept as concept,
        unnest.context as context,
        TRY_CAST(unnest.value AS FLOAT) as value,
        unnest.unit
    FROM (
        SELECT 
            company_number,
            UNNEST(financials) as filing
        FROM read_parquet('company_details/*.parquet', union_by_name=True)
        WHERE financials IS NOT NULL
    ),
    UNNEST(filing.facts) AS unnest
    WHERE unnest.value IS NOT NULL
)
SELECT 
    company_number,
    filing_type,
    filing_date,
    period,
    concept,
    context,
    value,
    unit
FROM financial_facts
WHERE period IS NOT NULL
ORDER BY company_number, filing_date DESC, period DESC, concept ASC, context ASC;
"""

# Execute query and convert to DataFrame
df_financials = con.execute(query).df()

print("\nColumns in the final dataset:")
print(df_financials.columns.tolist())

if len(df_financials) > 0:
    print("\nFirst few rows of data:")
    display(df_financials.head())
    
    # Calculate summary statistics for financial amounts
    print("\nSummary by financial concept:")
    concept_summary = df_financials.groupby('concept').agg({
        'company_number': 'nunique',
        'value': ['count', 'mean', 'median', 'std', 'min', 'max']
    }).round(2)
    
    concept_summary.columns = ['num_companies', 'count', 'mean', 'median', 'std', 'min', 'max']
    display(concept_summary.sort_values('num_companies', ascending=False))
    
    # Show the distribution of units used
    print("\nUnits used for each concept:")
    unit_summary = pd.crosstab(df_financials['concept'], df_financials['unit'])
    display(unit_summary)
    
    # Create visualization for top concepts
    top_concepts = concept_summary.nlargest(10, 'count').index
    
    fig = px.box(
        df_financials[df_financials['concept'].isin(top_concepts)],
        x='concept',
        y='value',
        title='Distribution of Values for Top Financial Concepts'
    )
    fig.update_layout(
        xaxis_title="Financial Concept",
        yaxis_title="Value",
        xaxis={'tickangle': 45}
    )
    display(fig)

Examining facts structure:

Available columns in facts:


,company_number,facts,filing_date,filing_type,unnest
0,00220603,[{'concept': 'bus:UKCompaniesHouseRegisteredNu...,2024-12-19,AA,{'concept': 'bus:UKCompaniesHouseRegisteredNum...


Examining facts structure:

Available columns in facts:


,company_number,facts,filing_date,filing_type,unnest
0,00220603,[{'concept': 'bus:UKCompaniesHouseRegisteredNu...,2024-12-19,AA,{'concept': 'bus:UKCompaniesHouseRegisteredNum...



Columns in the final dataset:
['company_number', 'filing_type', 'filing_date', 'period', 'concept', 'context', 'value', 'unit']

First few rows of data:


...

,company_number,filing_type,filing_date,period,concept,context,value,unit
0,00220603,AA,2024-12-19,2024-03-31,core:AccumulatedDepreciationImpairmentProperty...,B,11022.0,GBP
1,00220603,AA,2024-12-19,2024-03-31,core:AccumulatedDepreciationImpairmentProperty...,B_GP_IO,10152.0,GBP
2,00220603,AA,2024-12-19,2024-03-31,core:AccumulatedDepreciationImpairmentProperty...,B_GP_IP,870.0,GBP
3,00220603,AA,2024-12-19,2024-03-31,core:CashBankOnHand,B,2751.0,GBP
4,00220603,AA,2024-12-19,2024-03-31,core:Creditors,B_GX_IG,1641.0,GBP



Summary by financial concept:


,num_companies,count,mean,median,std,min,max
concept,,,,,,,
core:Equity,140,3442,241962.078125,33321.00,629635.48,0.0,8771628.0
core:Creditors,139,1985,208820.546875,26885.00,498440.05,0.0,5032391.0
core:AverageNumberEmployeesDuringPeriod,137,1012,12.530000,2.00,114.41,0.0,2687.0
core:NetCurrentAssetsLiabilities,135,1331,208274.312500,36480.00,615327.97,0.0,8670432.0
core:CurrentAssets,135,1294,378188.625000,54659.00,1016148.60,0.0,12493292.0
...,...,...,...,...,...,...,...
e:FutureMinimumLeasePaymentsUnderNon-cancellableOperatingLeases,1,6,248197.687500,278480.00,134811.99,74621.0,410417.0
core:WorkInProgress,1,1,5876.000000,5876.00,NaN,5876.0,5876.0
h:DirectorRemunerationBenefitsIncludingPaymentsToThirdParties,1,2,475357.000000,475357.00,1849.79,474049.0,476665.0



Units used for each concept:


unit,GBP,pure,shares,xbrli:pure,xbrli:shares
concept,,,,,
aurep:AuditFeesExpenses,28,0,0,0,0
c:AccruedLiabilitiesDeferredIncome,4,0,0,0,0
c:AccumulatedAmortisationImpairmentIntangibleAssets,2,0,0,0,0
c:AccumulatedDepreciationImpairmentPropertyPlantEquipment,5,0,0,0,0
c:AccumulatedDepreciationNotIncludingImpairmentPropertyPlantEquipment,5,0,0,0,0
...,...,...,...,...,...
uk-gaap:TotalAssetsLessCurrentLiabilities,80,0,0,0,0
uk-gaap:TradeCreditorsWithinOneYear,12,0,0,0,0
uk-gaap:TradeDebtors,11,0,0,0,0


In [47]:
df_financials

,company_number,filing_type,filing_date,period,concept,context,value,unit
0,00220603,AA,2024-12-19,2024-03-31,core:AccumulatedDepreciationImpairmentProperty...,B,11022.0,GBP
1,00220603,AA,2024-12-19,2024-03-31,core:AccumulatedDepreciationImpairmentProperty...,B_GP_IO,10152.0,GBP
2,00220603,AA,2024-12-19,2024-03-31,core:AccumulatedDepreciationImpairmentProperty...,B_GP_IP,870.0,GBP
3,00220603,AA,2024-12-19,2024-03-31,core:CashBankOnHand,B,2751.0,GBP
4,00220603,AA,2024-12-19,2024-03-31,core:Creditors,B_GX_IG,1641.0,GBP
...,...,...,...,...,...,...,...,...
77065,01825158,AA,2015-09-28,2013-12-31,uk-gaap-pt:ProfitLossAccountReserve,previous-mud,22086.0,GBP
77066,01825158,AA,2015-09-28,2013-12-31,uk-gaap-pt:ShareCapitalAllottedCalledUpPaid,previous-mud-shareclass-ordinary-1,10400.0,GBP
77067,01825158,AA,2015-09-28,2013-12-31,uk-gaap-pt:ShareholderFunds,previous-mud,32486.0,GBP
77068,01825158,AA,2015-09-28,2013-12-31,uk-gaap-pt:StocksInventory,previous-mud,2270.0,GBP


In [48]:
str(df_financials["concept"].unique().tolist())

"['core:AccumulatedDepreciationImpairmentPropertyPlantEquipment', 'core:CashBankOnHand', 'core:Creditors', 'core:CurrentAssets', 'core:Debtors', 'core:Equity', 'core:NetCurrentAssetsLiabilities', 'core:OtherCreditors', 'core:OtherDebtors', 'core:OtherTaxationSocialSecurityPayable', 'core:PropertyPlantEquipment', 'core:PropertyPlantEquipmentGrossCost', 'core:TotalAssetsLessCurrentLiabilities', 'bus:UKCompaniesHouseRegisteredNumber', 'core:AdditionsOtherThanThroughBusinessCombinationsPropertyPlantEquipment', 'core:AverageNumberEmployeesDuringPeriod', 'core:IncreaseFromDepreciationChargeForYearPropertyPlantEquipment', 'core:OtherDisposalsDecreaseInDepreciationImpairmentPropertyPlantEquipment', 'core:OtherDisposalsPropertyPlantEquipment', 'frs-core:AccruedLiabilitiesDeferredIncome', 'frs-core:AccumulatedDepreciationImpairmentPropertyPlantEquipment', 'frs-core:AmountsOwedByDirectors', 'frs-core:CashBankOnHand', 'frs-core:Creditors', 'frs-core:CurrentAssets', 'frs-core:Debtors', 'frs-core:Eq

In [49]:
df_financials.to_csv("financials.csv", index=False)

In [27]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('financials.csv')

# Keywords to search for in concepts
keywords = ['cash', 'bank', 'turnover', 'revenue', 'employee', 'asset', 'liabilit', 'debt']

# Filter concepts containing these keywords
filtered_concepts = []
for concept in df['concept'].unique():
    if any(keyword.lower() in concept.lower() for keyword in keywords):
        filtered_concepts.append(concept)

print("Relevant financial concepts:")
for concept in sorted(filtered_concepts):
    if concept.startswith('core:'):
        print(concept)

Relevant financial concepts:
core:AccruedLiabilities
core:AccruedLiabilitiesDeferredIncome
core:AccruedLiabilitiesNotExpressedWithinCreditorsSubtotal
core:AccumulatedAmortisationImpairmentIntangibleAssets
core:AdditionsOtherThanThroughBusinessCombinationsIntangibleAssets
core:AmortisationExpenseIntangibleAssets
core:AmortisationImpairmentExpenseIntangibleAssets
core:AmortisationRateUsedForIntangibleAssets
core:AverageNumberEmployeesDuringPeriod
core:BalancesWithBanks
core:BankBorrowings
core:BankBorrowingsOverdrafts
core:BankOverdrafts
core:CalledUpShareCapitalNotPaidNotExpressedAsCurrentAsset
core:CashBankOnHand
core:CashCashEquivalents
core:CashCashEquivalentsCashFlowValue
core:CashOnHand
core:CurrentAssetInvestments
core:CurrentAssets
core:Debtors
core:DeferredTaxAssetDebtors
core:DeferredTaxLiabilities
core:DisposalsDecreaseInAmortisationImpairmentIntangibleAssets
core:DisposalsIntangibleAssets
core:FinanceLeaseLiabilitiesPresentValueTotal
core:FixedAssets
core:FurtherItemCashFlowF

In [ ]:
core:AccruedLiabilities
core:AccruedLiabilitiesDeferredIncome
core:AccruedLiabilitiesNotExpressedWithinCreditorsSubtotal
core:AccumulatedAmortisationImpairmentIntangibleAssets
core:AdditionsOtherThanThroughBusinessCombinationsIntangibleAssets
core:AmortisationExpenseIntangibleAssets
core:AmortisationImpairmentExpenseIntangibleAssets
core:AmortisationRateUsedForIntangibleAssets
core:AverageNumberEmployeesDuringPeriod
core:BalancesWithBanks
core:BankBorrowings
core:BankBorrowingsOverdrafts
core:BankOverdrafts
core:CalledUpShareCapitalNotPaidNotExpressedAsCurrentAsset
core:CashBankOnHand
core:CashCashEquivalents
core:CashCashEquivalentsCashFlowValue
core:CashOnHand
core:CurrentAssetInvestments
core:CurrentAssets
core:Debtors
core:DeferredTaxAssetDebtors
core:DeferredTaxLiabilities
core:DisposalsDecreaseInAmortisationImpairmentIntangibleAssets
core:DisposalsIntangibleAssets
core:FinanceLeaseLiabilitiesPresentValueTotal
core:FixedAssets
core:FurtherItemCashFlowFromUsedInFinancingActivitiesComponentNetCashFlowsFromUsedInFinancingActivities
core:GainLossInCashFlowsFromChangeInCreditorsTradeOtherPayables
core:GainLossInCashFlowsFromChangeInDebtorsTradeOtherReceivables
core:IncreaseDecreaseDueToTransfersBetweenClassesIntangibleAssets
core:IncreaseDecreaseInCashCashEquivalentsBeforeForeignExchangeDifferencesChangesInConsolidation
core:IncreaseDecreaseInDeferredTaxLiabilityFromAmountRecognisedInProfitOrLoss
core:IncreaseDecreaseInNetDeferredTaxLiabilityFromAmountRecognisedInProfitOrLoss
core:IncreaseFromAmortisationChargeForYearIntangibleAssets
core:IncreaseFromImpairmentLossRecognisedInOtherComprehensiveIncomeIntangibleAssets
core:IncreaseFromImpairmentLossRecognisedInProfitOrLossIntangibleAssets
core:IntangibleAssets
core:IntangibleAssetsGrossCost
core:InterestIncomeOnBankDeposits
core:InvestmentsFixedAssets
core:NetAssetsLiabilities
core:NetAssetsLiabilitiesSubsidiaries
core:NetCashGeneratedFromOperations
core:NetCurrentAssetsLiabilities
core:NetDeferredTaxLiabilityAsset
core:OtherDebtors
core:OtherDisposalsDecreaseInAmortisationImpairmentIntangibleAssets
core:OtherDisposalsIntangibleAssets
core:OtherIncreaseDecreaseInAmortisationImpairmentIntangibleAssets
core:PrepaymentsAccruedIncomeNotExpressedWithinCurrentAssetSubtotal
core:PropertyPlantEquipmentIncludingRight-of-useAssets
core:ProvisionsForLiabilitiesBalanceSheetSubtotal
core:PurchaseIntangibleAssets
core:Share-basedPaymentExpenseCashSettled
core:StaffCostsEmployeeBenefitsExpense
core:TaxDecreaseIncreaseFromEffectRevenueExemptFromTaxation
core:TotalAdditionsIncludingFromBusinessCombinationsIntangibleAssets
core:TotalAssets
core:TotalAssetsLessCurrentLiabilities
core:TotalIncreaseDecreaseFromRevaluationsIntangibleAssets
core:TotalLiabilities
core:TradeDebtorsTradeReceivables
core:TransferToNon-currentAssetsOrDisposalGroupsHeldForSaleDecreaseInAmortisationImpairmentIntangibleAssets
core:TransferToNon-currentAssetsOrDisposalGroupsHeldForSaleDecreaseInDepreciationImpairmentPropertyPlantEquipment
core:TransfersToFromNon-currentAssetsOrDisposalGroupsHeldForSaleIntangibleAssets
core:TransfersToFromNon-currentAssetsOrDisposalGroupsHeldForSalePropertyPlantEquipment
core:TurnoverRevenue
core:UsefulLifeIntangibleAssetsYears